In [6]:
import json
import os
import subprocess
# Repo + global constants -----------------------------------------------------
repo_root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"]).decode().strip()
os.chdir(repo_root)
print("cwd =", os.getcwd())
from pathlib import Path

import numpy as np
import polars as pl
import torch
from itables import show as itshow
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from torch.utils.data import DataLoader, TensorDataset, random_split

from build_3dert_test2 import surface_z
from models.VAE.default import UNetVAE
from models.unet import Unet, Unet3D
from train.encoder.train_unet import load_sigma_dataset


REPO_ROOT = Path(repo_root)
RUNS_ROOT = REPO_ROOT / "saved_runs" / "grid_search_full_2"
DATA_PATH = REPO_ROOT / "data" / "3dert_test2.npy"
LOG_EPS = 1e-6
SPLIT_SEED = 159753
SPARSE_ENCODER_CKPT = (
    REPO_ROOT
    / "saved_runs/grid_search_vae_2/_lr2e-04_beta5e-02_kl400_bs128_ep1200_bc8_lc4_sparse/checkpoints/best_val_loss.pt"
)
DENSE_ENCODER_CKPT = (
    REPO_ROOT
    / "saved_runs/grid_search_vae_2/_lr2e-04_beta5e-02_kl400_bs128_ep1200_bc8_lc4/checkpoints/best_val_loss.pt"
)
SURFACE_PARAMS = {"dx_main": 30.0, "dy_main": 15.0, "k": 0.0025}

DROP_COLS = {"run_dir", "batch_size", "epochs", "best_checkpoint", "final_checkpoint", "training_done"}

# Helper functions ------------------------------------------------------------
def normalize_bool(value):
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() == "true"


def find_checkpoint(run_dir: Path, names: tuple) -> Path | None:
    for name in names:
        path = run_dir / "checkpoints" / name
        if path.exists():
            return path
    return None


def load_encoder(device: torch.device, sparse: bool) -> UNetVAE:
    encoder_ckpt = SPARSE_ENCODER_CKPT if sparse else DENSE_ENCODER_CKPT
    with (encoder_ckpt.parents[1] / "hparams.json").open("r", encoding="utf-8") as f:
        enc_hparams = json.load(f)
    encoder = UNetVAE(base_channels=int(enc_hparams["bc"]), latent_channels=int(enc_hparams["lc"])).to(device)
    checkpoint = torch.load(encoder_ckpt, map_location="cpu")
    ema_state = checkpoint.get("ema_state_dict")
    if ema_state is not None:
        ema_model = AveragedModel(encoder, multi_avg_fn=get_ema_multi_avg_fn(0.999))
        ema_model.load_state_dict(ema_state)
        encoder.load_state_dict(ema_model.module.state_dict())
    else:
        encoder.load_state_dict(checkpoint["model_state_dict"])
    encoder.eval()
    for param in encoder.parameters():
        param.requires_grad_(False)
    return encoder


def build_test_loader(
    data_path: Path,
    sparse: bool,
    batch_size: int,
    device: torch.device,
):
    data = np.load(data_path, allow_pickle=True).item()
    x = torch.from_numpy(np.asarray(data["X"], dtype=np.float32)).to(device)
    y, xq, yq, zq = load_sigma_dataset(str(data_path), log_eps=LOG_EPS, sparse=sparse)
    y = y.to(device)
    n_samples, in_c, in_h, in_w = x.shape
    _, out_c, out_d, out_h, out_w = y.shape
    dataset = TensorDataset(x, y)
    train_size = int(0.8 * n_samples)
    val_size = int(0.1 * n_samples)
    test_size = n_samples - train_size - val_size
    _, _, test_ds = random_split(
        dataset,
        [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(SPLIT_SEED),
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False,
        drop_last=False,
    )
    xx, yy, zz = np.meshgrid(xq, yq, zq, indexing="xy")
    surf = surface_z(xx, yy, SURFACE_PARAMS)
    solid_mask = torch.from_numpy(np.transpose(zz <= surf, (2, 0, 1))).to(device=device, dtype=torch.bool)
    return test_loader, (in_c, in_h, in_w), (out_c, out_d, out_h, out_w), solid_mask


def build_model(hparams: dict, device: torch.device, encoder: UNetVAE, test_loader: DataLoader):
    sparse = normalize_bool(hparams.get("sparse", False))
    pixel_output = normalize_bool(hparams.get("pixel_output", False))
    _, input_shape, output_shape, _ = build_test_loader(
        data_path=DATA_PATH,
        sparse=sparse,
        batch_size=int(hparams["batch_size"]),
        device=device,
    )
    in_c, _, _ = input_shape
    out_c, out_d, out_h, out_w = output_shape
    unet_ch = int(hparams["unet_ch"])
    if pixel_output:
        return Unet3D(
            in_channels=in_c,
            out_channels=out_c,
            output_shape=(out_d, out_h, out_w),
            ch=unet_ch,
        ).to(device)
    batch = next(iter(test_loader))[1][:1].to(device)
    with torch.no_grad():
        mu, _ = encoder.encode(batch)
    latent_out_c, latent_out_h, latent_out_w = mu.shape[1:]
    return Unet(
        in_channels=in_c,
        out_channels=latent_out_c,
        output_shape=(latent_out_h, latent_out_w),
        ch=unet_ch,
        ch_mul=[1, 2, 2],
    ).to(device)


def relative_metrics(pred: torch.Tensor, target: torch.Tensor):
    pred_flat = pred.float().reshape(pred.shape[0], -1)
    target_flat = target.float().reshape(target.shape[0], -1)
    diff = pred_flat - target_flat
    rel_l2 = torch.linalg.vector_norm(diff, ord=2, dim=1) / torch.clamp(
        torch.linalg.vector_norm(target_flat, ord=2, dim=1), min=1e-12,
    )
    rel_l1 = torch.linalg.vector_norm(diff, ord=1, dim=1) / torch.clamp(
        torch.linalg.vector_norm(target_flat, ord=1, dim=1), min=1e-12,
    )
    return rel_l1, rel_l2


def relative_metrics_masked(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor):
    mask_flat = mask.reshape(1, -1).expand(pred.shape[0], -1)
    pred_flat = pred.float().reshape(pred.shape[0], -1)
    target_flat = target.float().reshape(target.shape[0], -1)
    pred_masked = pred_flat[mask_flat].reshape(pred.shape[0], -1)
    target_masked = target_flat[mask_flat].reshape(target.shape[0], -1)
    diff = pred_masked - target_masked
    rel_l2 = torch.linalg.vector_norm(diff, ord=2, dim=1) / torch.clamp(
        torch.linalg.vector_norm(target_masked, ord=2, dim=1), min=1e-12,
    )
    rel_l1 = torch.linalg.vector_norm(diff, ord=1, dim=1) / torch.clamp(
        torch.linalg.vector_norm(target_masked, ord=1, dim=1), min=1e-12,
    )
    return rel_l1, rel_l2


def summarize(values: list, prefix: str) -> dict:
    if not values:
        return {
            f"mean_{prefix}": None,
            f"median_{prefix}": None,
            f"min_{prefix}": None,
            f"max_{prefix}": None,
        }
    arr = np.asarray(values, dtype=np.float64)
    return {
        f"mean_{prefix}": float(arr.mean()),
        f"median_{prefix}": float(np.median(arr)),
        f"min_{prefix}": float(arr.min()),
        f"max_{prefix}": float(arr.max()),
    }


def evaluate_run(run_dir: Path, device: torch.device) -> dict:
    hparams_path = run_dir / "hparams.json"
    with hparams_path.open("r", encoding="utf-8") as f:
        hparams = json.load(f)

    sparse = normalize_bool(hparams.get("sparse", False))
    antipodal_loss = normalize_bool(hparams.get("antipodal_loss", False))
    pixel_output = normalize_bool(hparams.get("pixel_output", False))
    best_ckpt = find_checkpoint(run_dir, ("best_val.pt", "best_val_loss.pt"))
    final_ckpt = find_checkpoint(run_dir, ("final_model.pt", "final.pt"))

    row: dict = {
        "run_dir": str(run_dir),
        "save_dir": hparams.get("save_dir", run_dir.name),
        "sparse": sparse,
        "antipodal_loss": antipodal_loss,
        "pixel_output": pixel_output,
        "lr": float(hparams["lr"]),
        "batch_size": int(hparams["batch_size"]),
        "epochs": int(hparams["epochs"]),
        "unet_ch": int(hparams["unet_ch"]),
        "best_checkpoint": str(best_ckpt) if best_ckpt is not None else None,
        "final_checkpoint": str(final_ckpt) if final_ckpt is not None else None,
        "training_done": best_ckpt is not None and final_ckpt is not None,
        "status": "ok" if best_ckpt is not None and final_ckpt is not None else "training_not_done",
    }
    row.update(summarize([], "rel_l1"))
    row.update(summarize([], "rel_l2"))
    row.update(summarize([], "below_surface_rel_l1"))
    row.update(summarize([], "below_surface_rel_l2"))

    if best_ckpt is None or final_ckpt is None:
        return row

    test_loader, _, _, solid_mask = build_test_loader(
        data_path=DATA_PATH,
        sparse=sparse,
        batch_size=int(hparams["batch_size"]),
        device=device,
    )
    encoder = load_encoder(device=device, sparse=sparse)
    model = build_model(hparams=hparams, device=device, encoder=encoder, test_loader=test_loader)
    checkpoint = torch.load(best_ckpt, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    rel_l1_values: list = []
    rel_l2_values: list = []
    below_surface_rel_l1_values: list = []
    below_surface_rel_l2_values: list = []
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            pred = model(x_batch)
            if not pixel_output:
                pred = encoder.decode(pred)
            rel_l1, rel_l2 = relative_metrics(pred, y_batch)
            below_surface_rel_l1, below_surface_rel_l2 = relative_metrics_masked(pred, y_batch, solid_mask)
            rel_l1_values.extend(rel_l1.detach().cpu().tolist())
            rel_l2_values.extend(rel_l2.detach().cpu().tolist())
            below_surface_rel_l1_values.extend(below_surface_rel_l1.detach().cpu().tolist())
            below_surface_rel_l2_values.extend(below_surface_rel_l2.detach().cpu().tolist())

    row.update(summarize(rel_l1_values, "rel_l1"))
    row.update(summarize(rel_l2_values, "rel_l2"))
    row.update(summarize(below_surface_rel_l1_values, "below_surface_rel_l1"))
    row.update(summarize(below_surface_rel_l2_values, "below_surface_rel_l2"))
    return row


# Evaluation flow -------------------------------------------------------------
device = torch.device("cuda:1")
run_dirs = sorted(path.parent for path in RUNS_ROOT.rglob("hparams.json"))
print(f"Found {len(run_dirs)} run(s) under {RUNS_ROOT}")

rows = [evaluate_run(run_dir, device) for run_dir in run_dirs]
if not rows:
    raise SystemExit("No runs found.")

df = pl.DataFrame(rows).sort(
    by=["status", "mean_below_surface_rel_l2", "save_dir"],
    descending=[False, False, False],
    nulls_last=True,
)

display_df = df.drop([c for c in DROP_COLS if c in df.columns]).to_pandas()
itshow(display_df, classes="display compact", paging=False)

completed_df = df.filter(pl.col("status") == "ok")
metrics_to_report = (
    ("mean_below_surface_rel_l2", "Below-Surface Relative L2"),
    ("mean_below_surface_rel_l1", "Below-Surface Relative L1"),
    ("mean_rel_l2", "Relative L2"),
    ("mean_rel_l1", "Relative L1"),
)

best_entries = []
worst_entries = []

for sparse_value, dataset_label in ((True, "Sparse"), (False, "Dense")):
    dataset_df = completed_df.filter(pl.col("sparse") == sparse_value)
    if dataset_df.height == 0:
        continue
    for metric_col, metric_label in metrics_to_report:
        metric_df = dataset_df.filter(pl.col(metric_col).is_not_null())
        if metric_df.height == 0:
            continue
        best_row = metric_df.sort(metric_col).row(0, named=True)
        worst_row = metric_df.sort(metric_col, descending=True).row(0, named=True)
        best_entries.append((dataset_label, metric_label, metric_col, best_row))
        worst_entries.append((dataset_label, metric_label, metric_col, worst_row))


COLOR_RED = "\033[31m"
COLOR_PURPLE = "\033[35m"
COLOR_RESET = "\033[0m"

def print_entries(entries, label, color):
    counts = {}
    for _, _, _, row in entries:
        counts[row["save_dir"]] = counts.get(row["save_dir"], 0) + 1

    for dataset_label, metric_label, metric_col, row in entries:
        name = row["save_dir"]
        value = row[metric_col]
        name_display = (
            f"{color}{name}{COLOR_RESET}" if counts[name] > 1 else name
        )
        prefix = "best" if label == "best" else "worst"
        print(
            f"{dataset_label} {prefix} ({metric_label}): {name_display} "
            f"({metric_col}={value:.6f})"
        )

print_entries(best_entries, "best", COLOR_RED)
print_entries(worst_entries, "worst", COLOR_PURPLE)

cwd = /home/johnma/ert3d_test2
Found 10 run(s) under /home/johnma/ert3d_test2/saved_runs/grid_search_full_2


Loading ITables v2.7.1 from the internet... (need help?)


Sparse best (Below-Surface Relative L2): grid_search_full_2/_sparse_pixel_only_lr5e-04_bs64_ep800_unet_ch32_pixel (mean_below_surface_rel_l2=0.028205)
Sparse best (Below-Surface Relative L1): grid_search_full_2/_sparse_pixel_only_lr5e-04_bs64_ep800_unet_ch32_pixel (mean_below_surface_rel_l1=0.010212)
Sparse best (Relative L2): grid_search_full_2/_sparse_pixel_only_lr5e-04_bs64_ep800_unet_ch32_pixel (mean_rel_l2=0.017962)
Sparse best (Relative L1): grid_search_full_2/_sparse_pixel_only_lr5e-04_bs64_ep800_unet_ch32_pixel (mean_rel_l1=0.006830)
Dense best (Below-Surface Relative L2): grid_search_full_2/_dense_antipodal_lr5e-04_bs32_ep800_unet_ch32_pixel (mean_below_surface_rel_l2=0.030611)
Dense best (Below-Surface Relative L1): grid_search_full_2/_dense_pixel_only_lr5e-04_bs32_ep800_unet_ch32_pixel (mean_below_surface_rel_l1=0.010838)
Dense best (Relative L2): grid_search_full_2/_dense_antipodal_lr5e-04_bs32_ep800_unet_ch32_pixel (mean_rel_l2=0.019505)
Dense best (Relative L1): grid_sear